In [28]:
# check if the GPU is detected
# boiler plate code found here: https://hariesef.medium.com/beginner-guide-to-generative-ai-large-language-model-part-4-training-fine-tuning-model-using-peft-81256c25e989

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3070


In [2]:
# download TinyLlama 1.1B

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0", filename="config.json")

'/home/reggie/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json'

### Load and Format Data

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

raw_data = pd.read_csv('swim_dataset.csv')
raw_data

# split to train and eval
train_data, eval_data = train_test_split(raw_data, test_size=0.1, random_state=42)
len(train_data), len(eval_data)

(888, 99)

In [4]:
# define data format
def preprocess(example):
    return {
        "text": f"<|system|> You are Swim Instructor helping athletes <|user|> {example['Question']} <|assistant|> {example['Answer']}"
    }

In [5]:
# load dataset
from datasets import Dataset

# load dataset/preprocess
train_dataset = Dataset.from_pandas(train_data)
eval_dataset = Dataset.from_pandas(eval_data)
train_data = train_dataset.map(preprocess)
eval_data = eval_dataset.map(preprocess)

Map:   0%|          | 0/888 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [6]:
train_data['text'][0]

'<|system|> You are Swim Instructor helping athletes <|user|> Is it beneficial to use a snorkel during the Golf Ball Drill? <|assistant|> Yes, a snorkel allows swimmers to focus on hand entry without worrying about breathing.'

In [8]:
import wandb

wandb.init(
    project="swim_coach_llama",
    name="run-10epochs",
)

### Define Model and Generation Function

In [9]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [10]:
# load the 4-bit quantized model to save compute power while retaining 94.8-99.1% accuracy
from transformers import AutoTokenizer, AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig, EarlyStoppingCallback

def get_model_and_tokenizer(model_id):
    tokenizer = AutoProcessor.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    # store the weights (parameters) of the model in 4-bit to save v-ram
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True
    )  
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")    
    model.config.use_cache = False
    # keep tensors on one device
    model.config.pretraining_tp = 1
    return model, tokenizer

model, tokenizer = get_model_and_tokenizer(model_id)

In [11]:
# define a generation function that takes a model (used for Human Eval, BLEU and ROUGE)

def generate(model, prompt, max_length=300, temperature=0.7, top_p=0.9, do_sample=True):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### Evaluate Baseline using BLEU

In [12]:
from tqdm.notebook import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
def calculateBLEU(eval_set, model):
    references = []
    hypotheses = []
    
    for row in tqdm(eval_data):
        curr_query = ('<|system|> You are Swim Instructor helping athletes <|user|> ' + row['Question'])
        reference = row['text'].split()
        hypothesis = generate(model, curr_query).split()

        # skip over if LLM failed to generate
        if len(hypothesis) < 9:
            continue
    
        references.append(reference)
        hypotheses.append(hypothesis)
    
    corpus_bleu_score = corpus_bleu(references, hypotheses, smoothing_function=SmoothingFunction().method1)
    return corpus_bleu_score

In [13]:
print(f"Corpus BLEU Score: {calculateBLEU(eval_data, model):.4f}")

  0%|          | 0/99 [00:00<?, ?it/s]

Corpus BLEU Score: 0.0001


### Evaluate Baseline using ROUGE

In [14]:
from rouge import Rouge

def calculateROUGE(eval_set, model):
    references = []
    hypotheses = []

    for row in tqdm(eval_data):
        curr_query = ('<|system|> You are Swim Instructor helping athletes <|user|> ' + row['Question'])
        reference = row['text']
        hypothesis = generate(model, curr_query)

        # skip over if LLM failed to generate
        if len(hypothesis) < 9:
            continue

        references.append(reference)
        hypotheses.append(hypothesis)
    rouge = Rouge()
    scores = rouge.get_scores(hypotheses, references, avg=True)
    return scores

In [15]:
print(f"Corpus ROUGE Score: {calculateROUGE(eval_data, model)}")

  0%|          | 0/99 [00:00<?, ?it/s]

Corpus ROUGE Score: {'rouge-1': {'r': 0.5909753325841303, 'p': 0.5848440188531379, 'f': 0.5166644133618943}, 'rouge-2': {'r': 0.5036886612672008, 'p': 0.5132251508753564, 'f': 0.422506497957036}, 'rouge-l': {'r': 0.5866810182231865, 'p': 0.5827817799830552, 'f': 0.5139665854140889}}


### Test QA on Base Model

In [16]:
response = generate(model, "<|system|> You are Swim Instructor helping athletes <|user|> How can I improve my distance per stroke?", max_length=200)
print(response)

<|system|> You are Swim Instructor helping athletes <|user|> How can I improve my distance per stroke?


### Define LoRs Config and Training Regimine

In [17]:
from peft import LoraConfig, PeftModel

# LoRa config
peft_config = LoraConfig(
    r=64,    # size of low rank matricies
    lora_alpha=32,    # scaling factor
    lora_dropout=0.08,  # Regularization via dropout
    bias='none',    # don't edit the bias of original model
    task_type="CAUSAL_LM",    # tells Lora this is a generation model
)

In [18]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="swim_coach_llama_64",
    dataset_text_field="text",
    packing=False,  # Set to True if you want data packing
    max_seq_length=1024,    # max token length accepted, anything longer will be truncated
    per_device_train_batch_size=16,    # batch size (essentially 64 because of next line)
    gradient_accumulation_steps=4,    # accumulate gradients across 4 batches before back-prop
    optim="paged_adamw_32bit",    # define optimization function
    learning_rate=2e-4,    # learning rate
    lr_scheduler_type="cosine",    # learning rate scheduler
    save_strategy="epoch",   # defines where to save checkpoints
    save_steps=10,    # how often to save checkpoint
    logging_steps=1,     # how often to log info
    num_train_epochs=10,    # number of epochs
    max_steps=200,    # max number of training steps
    fp16=True,    # train on mixed precision (16bit floats)
    eval_strategy="steps",  # Evaluate regularly
    eval_steps=10,    # how often to evaluate
    save_total_limit=10,           # Keep only last N checkpoints
    metric_for_best_model="eval_loss",  # Use eval_loss to find the best model
    greater_is_better=False,      # Lower eval_loss is better
)
# Trainer with validation dataset
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],  
)

Converting train dataset to ChatML:   0%|          | 0/888 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/888 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/888 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/888 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/99 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [19]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss
10,1.986000,1.841805
20,1.361800,1.368560
30,1.261000,1.242654
40,1.254700,1.197600
50,1.156300,1.168993
60,1.158900,1.148694
70,1.146200,1.134614
80,1.057200,1.123532
90,1.071100,1.113086
100,1.050300,1.102496


TrainOutput(global_step=200, training_loss=1.17964542388916, metrics={'train_runtime': 367.9782, 'train_samples_per_second': 34.785, 'train_steps_per_second': 0.544, 'total_flos': 6260189810491392.0, 'train_loss': 1.17964542388916})

In [22]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    load_in_8bit=False, 
    device_map="auto", 
    trust_remote_code=True,
    use_flash_attention_2=False,
)

#modify the folder according to which checkpoint has the best eval_score! (lower is better)
model_path = "./swim_coach_llama_64/checkpoint-196"

peft_model = PeftModel.from_pretrained(model, model_path, from_transformers=True, device_map="auto")
model = peft_model.merge_and_unload()

In [25]:
response = generate(model, "<|system|> You are Swim Instructor helping athletes <|user|> How can I develop an early vertical forearm?", max_length=350)
print(response)

<|system|> You are Swim Instructor helping athletes <|user|> How can I develop an early vertical forearm? <|assistant|> Practice overhead pulls with a partner and focus on the position of the elbow.


### Calculate BLEU Score post fine-tuning

In [26]:
print(f"Corpus BLEU Score: {calculateBLEU(eval_data, model):.4f}")

  0%|          | 0/99 [00:00<?, ?it/s]

Corpus BLEU Score: 0.0002


In [27]:
print(f"Corpus ROUGE Score: {calculateROUGE(eval_data, model)}")

  0%|          | 0/99 [00:00<?, ?it/s]

Corpus ROUGE Score: {'rouge-1': {'r': 0.6424242961739516, 'p': 0.6455456359864399, 'f': 0.6402556415074535}, 'rouge-2': {'r': 0.5463186614991943, 'p': 0.538332402682994, 'f': 0.5380684859414521}, 'rouge-l': {'r': 0.6310840008646247, 'p': 0.6341106213975675, 'f': 0.6289108803912307}}


### Result Interpretation

After a final run through the data, all ROUGE scores (recall, precision, and F1) as well as BLEU.  These results suggest that with compute resources availible, fine-tuning a generative model with LoRa can be a viable solution to problems that require a large amount of compute resources.